# Entrenamiento en Google Colab — raptors-cnn

**Tesis de Maestría · Brian Fernández Báez · 2026**

Notebook listo para entrenar las 4 arquitecturas en Google Colab Pro (recomendado: GPU T4 o A100). Útil cuando tu RTX 3050 local se queda corta para batch grandes o ConvNeXt.

**Pasos:**
1. Subir tu dataset a Google Drive (carpeta `datos/processed/`)
2. Run all → entrena las 4 arquitecturas
3. Pesos se guardan en Drive automáticamente

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repositorio

In [ ]:
!git clone https://github.com/ZOMBIECRAFTIAN/raptors-cnn.git
%cd raptors-cnn/codigo/pytorch

## 3. Instalar dependencias

Colab ya trae PyTorch + CUDA. Solo agregamos lo demás.

In [ ]:
!pip install -q timm grad-cam albumentations scikit-learn matplotlib seaborn tqdm pyyaml

## 4. Montar Google Drive (donde está el dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Asume que subiste tu dataset a:
#   Mi Drive/raptors-cnn-dataset/processed/{train,val,test}/<especie>/
DATASET_DRIVE = '/content/drive/MyDrive/raptors-cnn-dataset/processed'

# Lo enlazamos a la ruta esperada por config.py
import os, shutil
PROJECT_DATA = '/content/raptors-cnn/datos/processed'
os.makedirs(os.path.dirname(PROJECT_DATA), exist_ok=True)
if os.path.exists(PROJECT_DATA):
    shutil.rmtree(PROJECT_DATA)
os.symlink(DATASET_DRIVE, PROJECT_DATA)
!ls /content/raptors-cnn/datos/processed/train/ | head

## 5. Verificar setup

In [ ]:
!python verify_setup.py

## 6. Entrenar las 4 arquitecturas

Cada una guarda los pesos en `outputs/checkpoints/best_<arch>_stage2.pt`.

Tiempo estimado en T4: ~30-40 min por arquitectura. En A100: ~10-15 min.

In [ ]:
# En Colab podemos subir el batch porque las GPUs tienen más VRAM (T4=15GB, A100=40GB)
# Editamos config.py al vuelo:
import config
config.BATCH_SIZE = 64   # T4 aguanta esto
config.GRADIENT_ACCUM_STEPS = 1
print(f'BATCH_SIZE = {config.BATCH_SIZE}, AMP = {config.USE_AMP}')

In [ ]:
!python train.py --arch resnet50

In [ ]:
!mv outputs/checkpoints/best_stage2.pt outputs/checkpoints/best_resnet50_stage2.pt
!python train.py --arch efficientnet_b3

In [ ]:
!mv outputs/checkpoints/best_stage2.pt outputs/checkpoints/best_efficientnet_b3_stage2.pt
!python train.py --arch mobilenet_v3_large

In [ ]:
!mv outputs/checkpoints/best_stage2.pt outputs/checkpoints/best_mobilenet_v3_large_stage2.pt
!python train.py --arch convnext_tiny

## 7. Evaluar cada arquitectura

In [ ]:
for arch in ['resnet50', 'efficientnet_b3', 'mobilenet_v3_large', 'convnext_tiny']:
    print(f'\n========= EVAL {arch} =========')
    !python evaluate.py --arch {arch} --weights outputs/checkpoints/best_{arch}_stage2.pt

## 8. Guardar pesos en Drive para descargar a tu máquina local

In [ ]:
import shutil, os
DRIVE_OUT = '/content/drive/MyDrive/raptors-cnn-checkpoints'
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in os.listdir('outputs/checkpoints'):
    if f.endswith('.pt'):
        shutil.copy(f'outputs/checkpoints/{f}', f'{DRIVE_OUT}/{f}')
        print(f'✓ {f} copiado a Drive')
print('\nDescarga los .pt desde tu Google Drive a:')
print('  C:\\Users\\hogwa\\raptors-cnn\\codigo\\pytorch\\outputs\\checkpoints\\')